# Career Transition Data — IPUMS CPS

Extracts real-world job-to-job transitions from the IPUMS Current Population Survey,
maps Census occupation codes → SOC codes → O*NET titles, and produces an aggregate
transition frequency table ready to compare against our similarity scores.

---

## Step 0 — Download the data from IPUMS (one-time)

1. Go to **https://cps.ipums.org/cps/** and create a free account
2. Click **Select Data → Select Samples**
   - Check **ASEC** (Annual Social and Economic Supplement) for years **2018–2023**
   - Uncheck everything else
3. Click **Select Variables → Search** and add these variables:

   | Variable | Description |
   |---|---|
   | `YEAR` | Survey year |
   | `CPSIDP` | Person identifier (links records across months) |
   | `WTFINL` | Final person weight |
   | `EMPSTAT` | Employment status (current) |
   | `OCC` | Occupation (current, 2018 Census codes for 2020+) |
   | `OCCLY` | Occupation last year |
   | `IND` | Industry (current) |
   | `INDLY` | Industry last year |
   | `WKSWORK1` | Weeks worked last year |

4. Click **View Cart → Create Data Extract**
   - Format: **CSV**
   - Structure: **Rectangular (person)**
5. Submit and download when ready. Rename the file to:
   ```
   data/raw/ipums_cps_asec.csv.gz
   ```
   (IPUMS delivers a .gz file — leave it compressed, pandas reads it directly)

---

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')

IPUMS_FILE = RAW / 'ipums_cps_asec.csv.gz'

if not IPUMS_FILE.exists():
    raise FileNotFoundError(
        f"{IPUMS_FILE} not found.\n"
        "Follow the download instructions in the markdown cell above, "
        "then place the file at data/raw/ipums_cps_asec.csv.gz"
    )

## 1. Load IPUMS extract

In [ ]:
usecols = ['YEAR', 'CPSIDP', 'WTFINL', 'EMPSTAT', 'OCC', 'OCCLY', 'IND', 'INDLY', 'WKSWORK1']

raw = pd.read_csv(IPUMS_FILE, usecols=usecols, low_memory=False)
raw.columns = raw.columns.str.lower()

print(f"Loaded {len(raw):,} person-year records")
print(f"Years: {sorted(raw['year'].unique())}")
raw.head(3)

## 2. Filter to valid year-over-year transitions

In [ ]:
# EMPSTAT codes: 10=at work, 12=has job not at work last week
# OCC/OCCLY: 0 = N/A, 9999 = unknown
INVALID_OCC = {0, 9999}

transitions = raw[
    raw['empstat'].isin([10, 12]) &          # currently employed
    raw['occ'].notna() &
    raw['occly'].notna() &
    (~raw['occ'].isin(INVALID_OCC)) &
    (~raw['occly'].isin(INVALID_OCC)) &
    (raw['occ'] != raw['occly'])             # occupation actually changed
].copy()

# Zero-pad Census codes to 4 digits to match crosswalk
transitions['occ_str']   = transitions['occ'].astype(int).astype(str).str.zfill(4)
transitions['occly_str'] = transitions['occly'].astype(int).astype(str).str.zfill(4)

print(f"{len(transitions):,} person-years with a documented occupation change")

## 3. Map Census OCC codes → SOC codes

In [ ]:
xwalk = pd.read_csv(RAW / 'census_occ_to_soc_2018.csv', dtype=str)

# SOC codes in crosswalk are 6-digit (15-1252); O*NET uses 10-digit (15-1252.00)
# We'll match on the 6-digit prefix
xwalk['soc_6'] = xwalk['soc_code_6digit'].str[:7]  # e.g. '15-1252'

code_to_soc = xwalk.set_index('census_occ_code')['soc_6'].to_dict()
code_to_title = xwalk.set_index('census_occ_code')['census_title'].to_dict()

transitions['soc_to']   = transitions['occ_str'].map(code_to_soc)
transitions['soc_from'] = transitions['occly_str'].map(code_to_soc)

mapped = transitions.dropna(subset=['soc_to', 'soc_from'])
print(f"{len(mapped):,} transitions with both SOC codes mapped ({len(mapped)/len(transitions):.1%} of total)")

## 4. Aggregate transition counts (weighted)

In [ ]:
# Weighted count: each person record carries a survey weight representing real workers
agg = (
    mapped
    .groupby(['soc_from', 'soc_to'])['wtfinl']
    .sum()
    .reset_index()
    .rename(columns={'wtfinl': 'weighted_count'})
)

# Add human-readable titles from crosswalk
occ_to_census_title = {v: code_to_title[k] for k, v in code_to_soc.items()}
agg['title_from'] = agg['soc_from'].map(occ_to_census_title)
agg['title_to']   = agg['soc_to'].map(occ_to_census_title)

# Compute transition probability: given a 'from' job, what % moved to each 'to' job?
totals = agg.groupby('soc_from')['weighted_count'].transform('sum')
agg['transition_prob'] = (agg['weighted_count'] / totals).round(4)

agg = agg.sort_values('weighted_count', ascending=False)

print(f"{len(agg):,} unique (from → to) occupation pairs")
print(f"{agg['soc_from'].nunique():,} unique source occupations")
print(f"\nTop 20 most common transitions:")
agg[['title_from', 'title_to', 'weighted_count', 'transition_prob']].head(20)

## 5. Join with O*NET similarity scores

In [ ]:
# Load the O*NET master for SOC → full title mapping
master = pd.read_parquet(PROCESSED / 'onet_master.parquet')[['soc_code', 'title']]

# O*NET SOC codes are 10-digit (15-1252.00); match on the 7-char prefix
master['soc_6'] = master['soc_code'].str[:7]
soc6_to_onet = master.drop_duplicates('soc_6').set_index('soc_6')['soc_code'].to_dict()
soc6_to_title = master.drop_duplicates('soc_6').set_index('soc_6')['title'].to_dict()

agg['onet_soc_from'] = agg['soc_from'].map(soc6_to_onet)
agg['onet_soc_to']   = agg['soc_to'].map(soc6_to_onet)
agg['onet_title_from'] = agg['soc_from'].map(soc6_to_title)
agg['onet_title_to']   = agg['soc_to'].map(soc6_to_title)

# Load cumulative similarity matrix
import sys
sys.path.insert(0, '../src')
from similarity import compute_similarity_matrices

full_master = pd.read_parquet(PROCESSED / 'onet_master.parquet')
matrices = compute_similarity_matrices(full_master)
cum_sim = matrices['cumulative']

# Look up similarity score for each observed transition
def lookup_similarity(row):
    f, t = row['onet_soc_from'], row['onet_soc_to']
    if pd.isna(f) or pd.isna(t):
        return np.nan
    if f in cum_sim.index and t in cum_sim.columns:
        return round(cum_sim.loc[f, t], 4)
    return np.nan

agg['onet_similarity'] = agg.apply(lookup_similarity, axis=1)

matched = agg.dropna(subset=['onet_soc_from', 'onet_soc_to', 'onet_similarity'])
print(f"{len(matched):,} transitions with O*NET similarity scores matched")

## 6. Explore: do people move to similar jobs?

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of similarity scores for observed transitions
axes[0].hist(matched['onet_similarity'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(matched['onet_similarity'].mean(), color='red', linestyle='--',
                label=f"Mean: {matched['onet_similarity'].mean():.3f}")
axes[0].set_title('Similarity of Observed Career Transitions')
axes[0].set_xlabel('O*NET Cosine Similarity')
axes[0].set_ylabel('Number of transition pairs')
axes[0].legend()

# Weighted: higher-volume transitions tend to be more/less similar?
axes[1].scatter(
    matched['onet_similarity'],
    np.log1p(matched['weighted_count']),
    alpha=0.3, s=10, color='steelblue'
)
axes[1].set_title('Transition Volume vs. Similarity')
axes[1].set_xlabel('O*NET Cosine Similarity')
axes[1].set_ylabel('log(weighted count)')

plt.tight_layout()
plt.savefig(PROCESSED / 'transitions_vs_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Surprising transitions: high volume but low similarity (unexpected moves)
print("HIGH VOLUME, LOW SIMILARITY (unexpected transitions):")
surprising = matched[
    (matched['weighted_count'] > matched['weighted_count'].quantile(0.75)) &
    (matched['onet_similarity'] < matched['onet_similarity'].quantile(0.25))
].sort_values('weighted_count', ascending=False)
print(surprising[['onet_title_from', 'onet_title_to', 'weighted_count', 'onet_similarity']].head(15).to_string(index=False))

print("\nHIGH VOLUME, HIGH SIMILARITY (expected transitions):")
expected = matched[
    (matched['weighted_count'] > matched['weighted_count'].quantile(0.75)) &
    (matched['onet_similarity'] > matched['onet_similarity'].quantile(0.75))
].sort_values('weighted_count', ascending=False)
print(expected[['onet_title_from', 'onet_title_to', 'weighted_count', 'onet_similarity']].head(15).to_string(index=False))

In [ ]:
# For a specific role: real transitions vs top similarity predictions
FOCUS_TITLE = 'Software Developers'  # change to any O*NET title

focus_soc = full_master[full_master['title'].str.lower() == FOCUS_TITLE.lower()]['soc_code'].iloc[0]
focus_soc6 = focus_soc[:7]

# Real transitions FROM this role (CPS)
real_from = matched[matched['soc_from'] == focus_soc6].sort_values('weighted_count', ascending=False)

# Top predicted similar roles (O*NET)
from similarity import most_similar
predicted = most_similar(focus_soc, matrices, full_master, top_n=20)
predicted_soc6 = set(predicted.index.str[:7])

print(f"\n=== Real transitions FROM: {FOCUS_TITLE} ({focus_soc}) ===")
real_from['in_predicted'] = real_from['soc_to'].isin(predicted_soc6)
print(real_from[['onet_title_to', 'weighted_count', 'transition_prob', 'onet_similarity', 'in_predicted']]
      .head(20).to_string(index=False))

print(f"\n{real_from['in_predicted'].sum()} of top {len(real_from)} real transitions appear in top-20 predicted similar roles")

## 7. Save

In [ ]:
agg.to_parquet(PROCESSED / 'career_transitions.parquet', index=False)
matched.to_parquet(PROCESSED / 'transitions_with_similarity.parquet', index=False)
print("Saved:")
print("  data/processed/career_transitions.parquet      — all transitions")
print("  data/processed/transitions_with_similarity.parquet — with O*NET similarity joined")